In [1]:
# ============================================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ И ИМПОРТ
# ============================================================

!pip install -q ultralytics
!pip install -q "polars>=1.28,<1.33"

import os
import json
import random
from pathlib import Path
from collections import defaultdict

import yaml
import pandas as pd
from PIL import Image
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.6 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ============================================================
# FIX: Патч read_results_csv — polars → pandas
# ============================================================

from ultralytics.engine.trainer import BaseTrainer

def _read_results_csv_pandas(self):
    csv_path = self.csv
    if csv_path.exists() and csv_path.stat().st_size > 0:
        try:
            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]
            return {col: df[col].tolist() for col in df.columns}
        except Exception:
            return {}
    return {}

BaseTrainer.read_results_csv = _read_results_csv_pandas

In [3]:
# ============================================================
# 2. НАСТРОЙКА ПУТЕЙ
# ============================================================

BASE_PATH   = Path("/kaggle/input/competitions/dl-lab-2-stuff-detection")
DATASET_SRC = BASE_PATH / "yolo_dataset" / "yolo_dataset"
TEST_IMAGES = BASE_PATH / "test_images"  / "test_images"
SAMPLE_SUB  = BASE_PATH / "sample_sub.csv"

WORK        = Path("/kaggle/working")
DATASET_DIR = WORK / "dataset"
RUNS_DIR    = WORK / "runs"

In [4]:
# ============================================================
# 3. ПОДСЧЁТ СОТРУДНИКОВ В LABEL-ФАЙЛЕ
# ============================================================

STAFF_CLASS = 1

def count_staff(label_path):
    """Вернуть число объектов класса STAFF_CLASS в label-файле."""
    if not label_path.exists():
        return 0
    text = label_path.read_text().strip()
    if not text:
        return 0
    return sum(
        1 for line in text.splitlines()
        if len(line.split()) >= 5 and int(line.split()[0]) == STAFF_CLASS
    )

In [5]:
# ============================================================
# 4. СТРАТИФИЦИРОВАННЫЙ TRAIN / VAL SPLIT
#    Группируем изображения по кол-ву сотрудников (0, 1, 2, …).
#    Из каждой группы 15 % уходит в val → пропорции сохранены.
# ============================================================

def create_stratified_split(source_root, dest_root,
                            val_ratio=0.15, seed=42):
    src_imgs = source_root / "train" / "images"
    src_lbls = source_root / "train" / "labels"

    all_images = sorted(
        f for f in src_imgs.iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    )

    # ---------- группировка ----------
    groups = defaultdict(list)
    for img in all_images:
        n = count_staff(src_lbls / f"{img.stem}.txt")
        groups[n].append(img)

    # ---------- стратифицированное разбиение ----------
    random.seed(seed)
    train_list, val_list = [], []

    # Для подробной статистики
    train_groups = defaultdict(int)
    val_groups = defaultdict(int)

    for k in sorted(groups):
        imgs = groups[k][:]
        random.shuffle(imgs)
        if len(imgs) <= 1:               # слишком мало — всё в train
            train_list.extend(imgs)
            train_groups[k] += len(imgs)
            continue
        n_val = max(1, int(len(imgs) * val_ratio))
        val_list.extend(imgs[:n_val])
        train_list.extend(imgs[n_val:])
        val_groups[k] += n_val
        train_groups[k] += len(imgs) - n_val

    # ---------- symlinks ----------
    for split, files in [("train", train_list), ("val", val_list)]:
        img_d = dest_root / split / "images"
        lbl_d = dest_root / split / "labels"
        img_d.mkdir(parents=True, exist_ok=True)
        lbl_d.mkdir(parents=True, exist_ok=True)

        for img in files:
            dst = img_d / img.name
            if not dst.exists():
                os.symlink(str(img.resolve()), str(dst))
            lbl = src_lbls / f"{img.stem}.txt"
            if lbl.exists():
                dl = lbl_d / lbl.name
                if not dl.exists():
                    os.symlink(str(lbl.resolve()), str(dl))

    # ---------- подробный вывод ----------
    all_keys = sorted(set(list(train_groups.keys()) + list(val_groups.keys())))

    print("=" * 60)
    print("РЕЗУЛЬТАТ СТРАТИФИЦИРОВАННОГО SPLIT")
    print("=" * 60)
    print(f"  Всего изображений:  {len(all_images)}")
    print(f"  Train изображений:  {len(train_list)}")
    print(f"  Val   изображений:  {len(val_list)}")
    print()
    print(f"  {'Кол-во staff':<16} {'Train':>8} {'Val':>8} {'Всего':>8} {'Val %':>8}")
    print(f"  {'—' * 52}")
    for k in all_keys:
        t = train_groups[k]
        v = val_groups[k]
        total = t + v
        pct = f"{v / total * 100:.1f}%" if total > 0 else "—"
        print(f"  {k:<16} {t:>8} {v:>8} {total:>8} {pct:>8}")
    print(f"  {'—' * 52}")
    print(f"  {'ИТОГО':<16} {len(train_list):>8} {len(val_list):>8} "
          f"{len(all_images):>8} "
          f"{round(len(val_list)/len(all_images)*100, 1):>7}%")
    print("=" * 60)

    return train_list

In [6]:
train_images = create_stratified_split(
    DATASET_SRC, DATASET_DIR, val_ratio=0.15, seed=42
)

РЕЗУЛЬТАТ СТРАТИФИЦИРОВАННОГО SPLIT
  Всего изображений:  3908
  Train изображений:  3323
  Val   изображений:  585

  Кол-во staff        Train      Val    Всего    Val %
  ————————————————————————————————————————————————————
  0                     697      122      819    14.9%
  1                    2035      359     2394    15.0%
  2                     428       75      503    14.9%
  3                      94       16      110    14.5%
  4                      68       12       80    15.0%
  5                       1        1        2    50.0%
  ————————————————————————————————————————————————————
  ИТОГО                3323      585     3908    15.0%


In [7]:
# ============================================================
# 5. УМНЫЙ ТАЙЛИНГ: 2 ЛУЧШИХ ТАЙЛА ИЗ СЛУЧАЙНЫХ ПОЗИЦИЙ
#
#    Для каждого изображения:
#    1. Генерируем 6 случайных тайлов 640×360
#    2. Считаем "полезность" каждого тайла (кол-во объектов внутри)
#    3. Берём 2 лучших (с наибольшим числом объектов)
#    4. Пустые тайлы (0 объектов) пропускаем
#    Итого: train ≈ ×2-2.5
# ============================================================

def create_smart_tiles(train_images, source_root, dest_root,
                       tile_w=640, tile_h=360,
                       n_candidates=6, n_keep=2,
                       min_vis=0.30, seed=42):

    src_lbls = source_root / "train" / "labels"
    dst_imgs = dest_root / "train" / "images"
    dst_lbls = dest_root / "train" / "labels"

    rng = random.Random(seed)
    n_tiles = 0
    n_skipped = 0
    n_orig = len(train_images)

    for img_path in train_images:
        pil = Image.open(str(img_path))
        W, H = pil.size

        # Чтение labels
        lbl_path = src_lbls / f"{img_path.stem}.txt"
        boxes = []
        if lbl_path.exists():
            text = lbl_path.read_text().strip()
            if text:
                for ln in text.splitlines():
                    p = ln.split()
                    if len(p) < 5: continue
                    boxes.append((int(p[0]),
                                  float(p[1])*W, float(p[2])*H,
                                  float(p[3])*W, float(p[4])*H))

        # Если нет аннотаций — пропускаем тайлинг (оригинал уже в train)
        if not boxes:
            pil.close()
            n_skipped += 1
            continue

        # Генерируем случайные позиции тайлов
        candidates = []
        for _ in range(n_candidates):
            x0 = rng.randint(0, max(0, W - tile_w))
            y0 = rng.randint(0, max(0, H - tile_h))
            xe = min(x0 + tile_w, W)
            ye = min(y0 + tile_h, H)
            tw = xe - x0
            th = ye - y0

            # Считаем объекты внутри тайла
            tile_lines = []
            for c, cx, cy, bw, bh in boxes:
                bx1, by1 = cx - bw/2, cy - bh/2
                bx2, by2 = cx + bw/2, cy + bh/2
                cx1, cy1 = max(bx1, x0), max(by1, y0)
                cx2, cy2 = min(bx2, xe), min(by2, ye)
                cw_, ch_ = cx2 - cx1, cy2 - cy1
                if cw_ <= 2 or ch_ <= 2: continue
                if bw*bh > 0 and (cw_*ch_)/(bw*bh) < min_vis: continue
                ncx = max(0, min(1, ((cx1+cx2)/2 - x0) / tw))
                ncy = max(0, min(1, ((cy1+cy2)/2 - y0) / th))
                nw  = max(0.001, min(1, cw_ / tw))
                nh  = max(0.001, min(1, ch_ / th))
                tile_lines.append(f"{c} {ncx:.6f} {ncy:.6f} {nw:.6f} {nh:.6f}")

            if tile_lines:  # пропускаем пустые тайлы
                candidates.append((len(tile_lines), x0, y0, xe, ye, tw, th, tile_lines))

        # Сортируем по количеству объектов, берём лучшие
        candidates.sort(key=lambda x: x[0], reverse=True)
        selected = candidates[:n_keep]

        for i, (n_obj, x0, y0, xe, ye, tw, th, tile_lines) in enumerate(selected):
            crop = pil.crop((x0, y0, xe, ye))
            name = f"{img_path.stem}_t{i}"
            crop.save(str(dst_imgs / f"{name}.jpg"), quality=95)
            (dst_lbls / f"{name}.txt").write_text("\n".join(tile_lines))
            n_tiles += 1

        pil.close()

    total = len([f for f in dst_imgs.iterdir()
                 if f.suffix.lower() in {".jpg", ".jpeg", ".png"}])

    print()
    print("=" * 60)
    print("РЕЗУЛЬТАТ ТАЙЛИНГА")
    print("=" * 60)
    print(f"  Оригиналов в train (до тайлинга):   {n_orig}")
    print(f"  Тайлов создано:                     {n_tiles}")
    print(f"  Всего в train (оригиналы + тайлы):  {total}")
    print(f"  Увеличение:                         ×{total / n_orig:.1f}")
    print("=" * 60)

In [8]:
create_smart_tiles(train_images, DATASET_SRC, DATASET_DIR)


РЕЗУЛЬТАТ ТАЙЛИНГА
  Оригиналов в train (до тайлинга):   3323
  Тайлов создано:                     13292
  Всего в train (оригиналы + тайлы):  16615
  Увеличение:                         ×5.0


In [9]:
# ============================================================
# 6. ИСПРАВЛЕНИЕ data.yaml
# ============================================================

def fix_data_yaml(source_dir, new_root, output):
    for c in [source_dir / "train" / "data.yaml",
              source_dir / "data.yaml"]:
        if c.exists():
            orig = c
            break
    else:
        raise FileNotFoundError("data.yaml не найден")

    with open(orig) as f:
        cfg = yaml.safe_load(f)

    cfg["path"] = str(new_root)
    cfg["val"]  = "val/images"

    with open(output, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    print(f"data.yaml → {output}")
    return str(output)


yaml_path = fix_data_yaml(DATASET_SRC, DATASET_DIR, WORK / "data.yaml")

data.yaml → /kaggle/working/data.yaml


In [10]:
# ============================================================
# 7. ОБУЧЕНИЕ МОДЕЛИ  (YOLO11m, imgsz=640)
# ============================================================

def train_model(data_yaml, model_name,
                epochs, imgsz, batch,
                project="runs", run_name="train", seed=42):

    model = YOLO(model_name)

    model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project=project,
        name=run_name,
        exist_ok=True,

        device=0,        # используем только первый T4
        amp=True,        # Mixed Precision — максимум от Tensor Cores

        # ── Сохранение и ранняя остановка ──
        save=True,
        save_period=5,
        patience=10,

        # ── Оптимизатор ──
        optimizer="AdamW",
        lr0=0.0008,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,

        # ── Веса loss ──
        box=7.5,
        cls=2.0,
        dfl=1.5,

        # ── Аугментации ──

        # Геометрические
        degrees=10.0,    
        translate=0.15,       
        scale=0.5,            
        shear=2.0,            
        perspective=0.0003,   
        fliplr=0.5,           
        flipud=0.0,       

        # Смешивание
        mixup=0.1,           
        erasing=0.1,         

        # Яркость (без изменения цвета)
        hsv_h=0.0,           
        hsv_s=0.0,     
        hsv_v=0.15,    

        # Отключено
        copy_paste=0.0,   

        # ── Прочее ──
        workers=4,
        seed=seed,
        verbose=True,
        val=True,
        plots=True,
    )
    return model

In [ ]:
model = train_model(
    data_yaml=yaml_path,
    model_name="yolo11m.pt",
    epochs=60,
    imgsz=640,
    batch=16,
    project="train_yolo11m",
    seed=42,
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.15, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.15, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.000

In [ ]:
# ============================================================
# 7. ЗАГРУЗКА ЛУЧШИХ ВЕСОВ
# ============================================================

best_pt = RUNS_DIR / "train" / "weights" / "best.pt"
last_pt = RUNS_DIR / "train" / "weights" / "last.pt"
weights = best_pt if best_pt.exists() else last_pt
print(f"Веса: {weights}")

In [ ]:
# ============================================================
# 4. SAHI ИНФЕРЕНС
# ============================================================

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=str(WEIGHTS),
    confidence_threshold=0.25,
    device="cuda:0",
    image_size=640,
)

IMG_W, IMG_H = 1280, 720
sample_sub = pd.read_csv(str(SAMPLE_SUB))

rows = []
total_boxes = 0

for idx, row in sample_sub.iterrows():
    img_name = str(row["image_name"])
    img_path = str(TEST_IMAGES / img_name)

    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=480,
        slice_width=640,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        perform_standard_pred=True,
        postprocess_type="NMS",
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.5,
        verbose=0,
    )

    boxes = []
    for pred in result.object_prediction_list:
        if pred.category.name != STAFF_NAME:
            continue

        conf = pred.score.value
        bbox = pred.bbox

        xc = (bbox.minx + bbox.maxx) / 2 / IMG_W
        yc = (bbox.miny + bbox.maxy) / 2 / IMG_H
        w  = (bbox.maxx - bbox.minx) / IMG_W
        h  = (bbox.maxy - bbox.miny) / IMG_H

        boxes.append([round(xc,6), round(yc,6),
                      round(w,6), round(h,6), round(conf,4)])

    total_boxes += len(boxes)
    rows.append({
        "id": idx,
        "image_name": img_name,
        "boxes": json.dumps(boxes, separators=(",",":")),
    })

    if (idx+1) % 200 == 0:
        print(f"  [{idx+1}/{len(sample_sub)}]")

sub = pd.DataFrame(rows)
sub.to_csv("submission_11m.csv", index=False)
imgs_with_det = sum(1 for r in rows if r["boxes"] != "[]")
print(f"Submission: {len(sub)} imgs, {imgs_with_det} w/ det, {total_boxes} boxes")